# Preprocessing — US License Plates

Notebook này thực hiện **yêu cầu 1 của đề midterm: clean – normalize – augment**, xử lý đúng các vấn đề mà EDA đã chỉ ra trên bộ gốc.

| Bước | Việc làm | Căn cứ từ EDA |
|---|---|---|
| 1. Gộp | Đọc ảnh của `train`, `valid`, `test` vào một bảng chung | `valid` và `test` gốc chỉ có 5 ảnh mỗi lớp |
| 2. Đánh dấu ảnh giống nhau | Gom các cặp ảnh giống nhau thành nhóm để xếp chung vào train; không xoá ảnh nào | 485 cặp ảnh có độ giống ≥ 0,70, 58 cặp nằm ở hai thư mục khác nhau |
| 3. Sửa nhãn | `HAWAI` → `HAWAII`, `MISSIPPI` → `MISSISSIPPI` | Sai ở tên thư mục và ở hai cột của `plates.csv` |
| 4. Chia lại | 70 / 15 / 15 theo từng bang, seed cố định | Cần tập đánh giá đủ lớn và không rò rỉ |
| 5. Ghi bộ mới | Thư mục sạch + `plates.csv` mới, kèm kiểm tra | |
| 6. Normalize | Tính mean / std trên train mới | |
| 7. Augment | Xoay, dịch, co giãn nhẹ; đổi độ sáng, tương phản; không lật | Biển luôn canh giữa và thẳng; chữ có hướng |
| 8. Nạp dữ liệu | `tf.data.Dataset` cho ba mô hình | |

**Đầu vào**

- `data/new_plates(raw_dataset)/` — bộ gốc, **chỉ đọc, không bị sửa**.
- `data/eda_outputs/duplicate_pairs.csv` — danh sách cặp ảnh giống nhau do notebook EDA tạo ra. Nếu chưa có file này, notebook tự tìm lại bằng đúng phương pháp của EDA.

**Đầu ra** (tất cả nằm trong `data/`)

- `data/new_plates(clean_dataset)/` — bộ đã làm sạch và chia lại, kèm `plates.csv` mới.
- `data/pre_outputs/` — hình minh hoạ của notebook này.
- `data/pre_outputs/preprocess_config.json` — mean, std, danh sách lớp, kích thước ảnh, để các notebook mô hình dùng lại.

**Cần cài sẵn:** `tensorflow` (dùng cho bước 7 và 8). Không cần GPU.

In [ ]:
%pip install -q pillow matplotlib pandas tqdm

In [ ]:
import os, json, shutil, hashlib, random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from tqdm.auto import tqdm
from IPython.display import display, Image as IPyImage

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ---- đường dẫn
RAW_DIR = "data/new_plates(raw_dataset)"        # bộ gốc (chỉ đọc)
EDA_DIR = "data/eda_outputs"                    # kết quả của notebook EDA
CLEAN_DIR = "data/new_plates(clean_dataset)"    # bộ sau tiền xử lý (được tạo mới mỗi lần chạy)
OUT = "data/pre_outputs"                        # hình minh hoạ của notebook này
CONFIG_PATH = "data/pre_outputs/preprocess_config.json"

# ---- tham số
SEED = 42
VALID_FRAC, TEST_FRAC = 0.15, 0.15
LABEL_FIXES = {"HAWAI": "HAWAII", "MISSIPPI": "MISSISSIPPI"}
IMG_H, IMG_W = 128, 224                         # giữ tỉ lệ 1,75 của biển số
BATCH_SIZE = 64

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
os.makedirs(OUT, exist_ok=True)
assert os.path.isdir(RAW_DIR), f"Không thấy thư mục bộ gốc '{RAW_DIR}' — sửa RAW_DIR"
assert os.path.abspath(CLEAN_DIR) != os.path.abspath(RAW_DIR), "CLEAN_DIR phải khác RAW_DIR"

C = dict(surface="#fcfcfb", text="#0b0b0b", text2="#52514e", grid="#e6e5e0",
         blue="#2a78d6", orange="#eb6834", aqua="#1baf7a")
plt.rcParams.update({
    "figure.facecolor": C["surface"], "axes.facecolor": C["surface"], "savefig.facecolor": C["surface"],
    "figure.dpi": 100, "savefig.dpi": 120, "savefig.bbox": "tight",
    "font.size": 10, "text.color": C["text"],
    "axes.edgecolor": C["grid"], "axes.labelcolor": C["text2"], "axes.titlecolor": C["text"],
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlepad": 12,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "axes.grid": True, "grid.color": C["grid"], "grid.linewidth": 0.8,
    "xtick.color": C["text2"], "ytick.color": C["text2"], "xtick.labelsize": 9, "ytick.labelsize": 9,
    "xtick.major.size": 0, "ytick.major.size": 0, "legend.frameon": False,
})

def save(fig, name, photo=False):
    if photo:
        path = os.path.join(OUT, name + ".jpg")
        fig.savefig(path, dpi=110, pil_kwargs={"quality": 85}); plt.close(fig)
        display(IPyImage(filename=path))
    else:
        fig.savefig(os.path.join(OUT, name + ".png")); plt.show()

def show_grid(images, caps, ncols, title, name, cell=1.9, lines=2):
    # images: đường dẫn hoặc ảnh PIL
    nrows = int(np.ceil(len(images) / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(cell * ncols, (cell * IMG_H / IMG_W + 0.22 + 0.14 * lines) * nrows + 0.6))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, im, cap in zip(axes, images, caps):
        ax.imshow(im if isinstance(im, PILImage.Image) else PILImage.open(im).convert("RGB"))
        ax.set_title(cap, fontsize=7.5, fontweight="normal", loc="center", pad=3, color=C["text2"])
    fig.suptitle(title, x=0.01, ha="left", fontsize=12, fontweight="bold", color=C["text"])
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.45 / fig.get_figheight()))
    save(fig, name, photo=True)

## 1. Gộp ba thư mục thành một bảng chung

Đọc toàn bộ ảnh của `train`, `valid`, `test` vào một bảng. Mỗi ảnh là một dòng, có:

- `key`: đường dẫn gốc dạng `split/bang/tên_file` (ví dụ `train/HAWAI/032.jpg`). Đây là "tên định danh" của ảnh, trùng với cách ghi trong `duplicate_pairs.csv` và trong `plates.csv` gốc.
- `split`, `label`: thư mục split và tên bang **như trong bộ gốc**. Các cột này được giữ nguyên để truy được nguồn; nhãn đã sửa sẽ nằm ở một cột mới (bước 3).

Từ đây trở đi mọi thao tác làm trên bảng này. Thư mục gốc không bị đụng tới.

In [ ]:
rows = []
for s in ("train", "valid", "test"):
    sdir = os.path.join(RAW_DIR, s)
    assert os.path.isdir(sdir), f"Không thấy thư mục {sdir}"
    for lab in sorted(os.listdir(sdir)):
        ldir = os.path.join(sdir, lab)
        if not os.path.isdir(ldir):
            continue
        for fn in sorted(os.listdir(ldir)):
            if fn.lower().endswith((".jpg", ".jpeg", ".png")):
                rows.append(dict(key=f"{s}/{lab}/{fn}", split=s, label=lab, name=fn, path=os.path.join(ldir, fn)))
df = pd.DataFrame(rows)
pos = {k: i for i, k in enumerate(df["key"])}          # key -> số thứ tự dòng

print(f"Bộ đã gộp: {len(df):,} ảnh · {df['label'].nunique()} lớp")
display(df["split"].value_counts().reindex(["train", "valid", "test"]).rename("số ảnh (theo thư mục gốc)").to_frame().T)
per_class = df.groupby("label").size()
print(f"Ảnh mỗi lớp trong bộ đã gộp: {per_class.min()} – {per_class.max()} (trung bình {per_class.mean():.1f})")

## 2. Đánh dấu ảnh giống nhau

EDA tìm thấy nhiều cặp ảnh rất giống nhau. Nếu hai ảnh như vậy rơi vào train và test thì mô hình đã "thấy trước đáp án" và kết quả test cao giả.

Danh sách cặp lấy từ EDA, nơi **mọi cặp ảnh cùng bang** đã được so kỹ (khoảng 679 nghìn cặp). Các cặp có độ giống **≥ 0,70** gồm cả hai trường hợp: cùng một ảnh bị lưu lại, cắt lại hoặc đổi màu (kể cả một cặp giống hệt từng pixel), và hai biển khác nhau cùng mẫu thiết kế. Hai trường hợp này đan xen ở cùng mức độ giống nên không tách được bằng ngưỡng. Vì vậy notebook **không xoá ảnh nào** mà xử lý chung cho cả hai:

- gom các cặp thành nhóm (A giống B, B giống C thì A, B, C cùng một nhóm);
- ở bước 4, mọi ảnh trong nhóm được **xếp chung vào train**.

Cách này an toàn ở cả hai trường hợp: nếu là ảnh trùng thật thì không rò rỉ sang valid / test, nếu là hai biển khác nhau thì không mất ảnh nào. Chỉ cần một ngưỡng (0,70), và ngưỡng đó không quyết định xoá ảnh nào.

In [ ]:
# ---- các cặp ảnh giống nhau (độ giống >= 0,70)
GROUP_T = 0.70

def find_similar_pairs(df):
    # Tìm lại cặp ảnh giống nhau bằng đúng phương pháp của EDA: so kỹ MỌI cặp ảnh cùng bang
    # (chỉ dùng khi chưa có kết quả EDA phù hợp)
    def unit_rows(v):
        v = v.reshape(len(v), -1).astype(np.float32)
        v = v - v.mean(1, keepdims=True)
        return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-6)

    def align_features(path):
        g = PILImage.open(path).convert("RGB").convert("L")
        center = np.asarray(g.resize((112, 64), PILImage.BILINEAR), dtype=np.float32)[8:-8, 12:-12]
        shifted = []
        for sc in (0.92, 0.96, 1.0, 1.04, 1.08):
            w, h = int(round(112 * sc)), int(round(64 * sc))
            B = np.asarray(g.resize((w, h), PILImage.BILINEAR), dtype=np.float32)
            for dy in range(-6, 7, 2):
                for dx in range(-8, 9, 2):
                    y0, x0 = (h - 48) // 2 + dy, (w - 88) // 2 + dx
                    if y0 >= 0 and x0 >= 0 and y0 + 48 <= h and x0 + 88 <= w:
                        shifted.append(B[y0:y0 + 48, x0:x0 + 88])
        return unit_rows(center[None])[0], unit_rows(np.stack(shifted))

    out = []
    labels_arr = df["label"].values
    for lab in tqdm(sorted(df["label"].unique()), desc="so kỹ từng bang"):
        idx = np.where(labels_arr == lab)[0]
        feats = [align_features(df["path"].iat[i]) for i in idx]
        A = np.stack([f[0] for f in feats]); V = feats[0][1].shape[0]
        S = np.empty((len(idx), len(idx)), dtype=np.float32)
        for j0 in range(0, len(idx), 16):
            Bv = np.concatenate([f[1] for f in feats[j0:j0 + 16]])
            S[:, j0:j0 + 16] = (A @ Bv.T).reshape(len(idx), -1, V).max(-1)
        xs, ys = np.where(np.triu(S, 1) >= GROUP_T)
        for x, y in zip(xs, ys):
            a, b = int(idx[x]), int(idx[y])
            out.append(dict(image_a=df["key"].iat[a], image_b=df["key"].iat[b], label=lab,
                            split_a=df["split"].iat[a], split_b=df["split"].iat[b], similarity=round(float(S[x, y]), 4)))
    return pd.DataFrame(out).sort_values("similarity", ascending=False).reset_index(drop=True)

dup_csv = os.path.join(EDA_DIR, "duplicate_pairs.csv")
eda_json = os.path.join(EDA_DIR, "eda_results.json")
eda_dup = {}
if os.path.exists(dup_csv) and os.path.exists(eda_json):
    with open(eda_json, encoding="utf-8") as f:
        eda_dup = json.load(f).get("duplicates", {})
# chỉ dùng danh sách của EDA nếu EDA đã so mọi cặp cùng bang với ngưỡng không cao hơn ngưỡng ở đây
if eda_dup.get("method") == "all_same_label_pairs" and eda_dup.get("group_threshold", 1) <= GROUP_T:
    pairs = pd.read_csv(dup_csv)
    print("Dùng danh sách cặp ảnh giống nhau của EDA:", dup_csv)
else:
    # chưa chạy EDA, hoặc kết quả EDA cũ (lọc bằng ảnh thu nhỏ / ngưỡng cao hơn) -> danh sách sẽ thiếu cặp
    print("Chưa có danh sách phù hợp trong", EDA_DIR, "→ tự tìm lại các cặp ảnh giống nhau")
    pairs = find_similar_pairs(df)
pairs = pairs[pairs["similarity"] >= GROUP_T].reset_index(drop=True)

unknown = (set(pairs["image_a"]) | set(pairs["image_b"])) - set(df["key"])
assert not unknown, f"Có ảnh trong danh sách không thuộc bộ gốc: {sorted(unknown)[:5]}"

class UnionFind:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[max(ra, rb)] = min(ra, rb)

# nối hai ảnh của mỗi cặp vào cùng một nhóm
grp_uf = UnionFind(len(df))
for r in pairs.itertuples():
    grp_uf.union(pos[r.image_a], pos[r.image_b])
df["unit"] = [grp_uf.find(i) for i in range(len(df))]
unit_size = df.groupby("unit").size()
df["in_group"] = df["unit"].map(unit_size) > 1
grouped_df = df[df["in_group"]]

cross = int((pairs["split_a"] != pairs["split_b"]).sum())
print(f"Cặp ảnh giống nhau (độ giống ≥ {GROUP_T}): {len(pairs)} · trong đó {cross} cặp nằm ở hai thư mục gốc khác nhau")
print(f"Số nhóm ảnh giống nhau: {(unit_size > 1).sum()} (nhóm lớn nhất gồm {unit_size.max()} ảnh)")
print(f"Ảnh thuộc các nhóm này, sẽ được xếp chung vào train: {len(grouped_df)} ({len(grouped_df) / len(df):.1%})")
print("Bang có nhiều ảnh như vậy nhất:", grouped_df["label"].value_counts().head(5).to_dict())

ex = pairs.sample(min(9, len(pairs)), random_state=SEED).sort_values("similarity", ascending=False)
imgs, caps = [], []
for r in ex.itertuples():
    for k in (r.image_a, r.image_b):
        imgs.append(df["path"].iat[pos[k]]); caps.append(f"{k}\nđộ giống {r.similarity:.3f}")
show_grid(imgs, caps, 6, "Ví dụ các cặp ảnh giống nhau (mỗi cặp hai ô liền nhau) — được giữ và xếp chung vào train",
          "pre01_similar_pairs")

> **Nhận xét**
>
> - Có **485 cặp ảnh giống nhau** (độ giống ≥ 0,70), trong đó 58 cặp nằm ở hai thư mục gốc khác nhau. Chúng gom thành 286 nhóm (nhóm lớn nhất 18 ảnh), gồm tổng cộng **693 ảnh** (7,9% bộ dữ liệu).
> - **Không ảnh nào bị xoá**: bộ dữ liệu vẫn đủ 8.721 ảnh. Cặp giống hệt từng pixel (`valid/OREGON/5.jpg` và `test/OREGON/1.jpg`) cũng nằm trong danh sách và sẽ cùng vào train.
> - Bang có nhiều ảnh thuộc các nhóm nhất là Wyoming (56 ảnh), rồi Arkansas (28). Đây là các bang có một mẫu biển xuất hiện rất nhiều lần.
> - Hệ quả phải chấp nhận: train có một số ảnh lặp lại (cùng một tấm biển xuất hiện hai lần). Chúng làm vài tấm biển được tính nặng hơn một chút khi huấn luyện, nhưng không làm sai kết quả đánh giá.

## 3. Sửa nhãn sai chính tả

EDA thấy `HAWAI` và `MISSIPPI` sai ở **ba chỗ** của bộ gốc: tên thư mục (ở cả ba split), cột `labels` và cột `filepaths` của `plates.csv`.

Vì dữ liệu đã được gộp vào một bảng, chỉ cần sửa **một lần trên một cột**: tạo cột `label_new` từ cột `label`. Ở bước 5, tên thư mục và hai cột của `plates.csv` mới đều được sinh ra từ `label_new`, nên ba chỗ tự khớp nhau.

Hai tên khác không khớp tên chuẩn là `CNMI` (viết tắt của Northern Mariana Islands) và `U S VIRGIN ISLANDS`; đó là cách đặt tên của bộ gốc chứ không phải lỗi chính tả, nên giữ nguyên.

In [ ]:
df["label_new"] = df["label"].replace(LABEL_FIXES)

fix = (df[df["label"].isin(LABEL_FIXES)].groupby(["label", "label_new", "split"]).size().unstack(fill_value=0)
         .reindex(columns=["train", "valid", "test"], fill_value=0))
fix["tổng"] = fix.sum(axis=1)
fix.index.names = ["nhãn gốc", "nhãn mới"]
display(fix)

CLASSES = sorted(df["label_new"].unique())
assert len(CLASSES) == df["label"].nunique(), "Sửa nhãn không được làm gộp hai lớp"
assert not set(LABEL_FIXES) & set(CLASSES)

# class id của hai lớp có đổi không? (id = thứ tự bảng chữ cái)
old_classes = sorted(df["label"].unique())
ids = pd.DataFrame({"class id cũ": {new: old_classes.index(old) for old, new in LABEL_FIXES.items()},
                    "class id mới": {new: CLASSES.index(new) for new in LABEL_FIXES.values()}})
display(ids)
print(f"Số lớp sau khi sửa: {len(CLASSES)}")

> **Nhận xét**
>
> - 154 ảnh `HAWAI` (144 / 5 / 5 ở ba thư mục gốc) nay mang nhãn `HAWAII`; 158 ảnh `MISSIPPI` (148 / 5 / 5) nay mang nhãn `MISSISSIPPI`.
> - Vẫn là 56 lớp. `class id` của hai lớp không đổi (13 và 26) vì tên mới giữ nguyên vị trí trong thứ tự bảng chữ cái.
> - Đây là lỗi tên lớp chứ không phải ảnh bị gán nhầm bang, nên việc sửa không ảnh hưởng tới mô hình; nó làm đúng tên hiển thị trong biểu đồ, confusion matrix và báo cáo.

## 4. Chia lại 70 / 15 / 15

Bộ gốc chia 8.161 / 280 / 280: `valid` và `test` chỉ có **5 ảnh mỗi lớp**, nên accuracy đo trên đó dao động rất mạnh. Vì vậy chia lại bộ đã gộp (sau bước 2).

Cách chia:

- **Stratified:** chia riêng trong từng bang, để cả 56 bang có mặt ở ba tập theo cùng tỉ lệ.
- **Theo nhóm:** mọi ảnh thuộc các nhóm ảnh giống nhau (bước 2) được xếp thẳng vào train, bảo đảm không có hai ảnh giống nhau nằm ở hai tập khác nhau.
- **Tái lập được:** thứ tự xáo trộn dùng `SEED = 42` (cộng số thứ tự của lớp), nên chạy lại luôn ra cùng một cách chia.

In [ ]:
kept = df.copy()                       # cột "unit" (nhóm ảnh giống nhau) đã có từ bước 2

assign, n_forced = {}, 0
for ci, c in enumerate(CLASSES):
    sub = kept[kept["label_new"] == c]
    units = {}
    for i, u in zip(sub.index, sub["unit"]):
        units.setdefault(u, []).append(i)
    grouped = [i for u in units.values() if len(u) > 1 for i in u]      # nhóm ảnh giống nhau -> train
    singles = [u[0] for u in units.values() if len(u) == 1]
    rng = np.random.RandomState(SEED + ci)
    singles = [singles[k] for k in rng.permutation(len(singles))]
    n_test = int(round(TEST_FRAC * len(sub)))
    n_valid = int(round(VALID_FRAC * len(sub)))
    for i in singles[:n_test]:
        assign[i] = "test"
    for i in singles[n_test:n_test + n_valid]:
        assign[i] = "valid"
    for i in singles[n_test + n_valid:] + grouped:
        assign[i] = "train"
    n_forced += len(grouped)
kept["split_new"] = kept.index.map(assign)

SPLITS = ["train", "valid", "test"]
counts = kept.groupby(["label_new", "split_new"]).size().unstack(fill_value=0).reindex(index=CLASSES, columns=SPLITS)
counts["tổng"] = counts.sum(axis=1)
total = counts[SPLITS].sum()
print("Số ảnh mỗi tập:", total.to_dict(), "· tỉ lệ (%):", (total / total.sum() * 100).round(1).to_dict())
print(f"Ảnh thuộc các nhóm ảnh giống nhau, xếp thẳng vào train: {n_forced}")
display(pd.DataFrame({s: {"ít nhất": counts[s].min(), "nhiều nhất": counts[s].max(), "trung bình": round(counts[s].mean(), 1)}
                      for s in SPLITS + ["tổng"]}))
print("Nguồn gốc ảnh (hàng: thư mục gốc, cột: tập mới):")
prov = pd.crosstab(kept["split"], kept["split_new"], margins=True, margins_name="tổng").reindex(
    index=SPLITS + ["tổng"], columns=SPLITS + ["tổng"])
prov.index.name, prov.columns.name = "thư mục gốc", "tập mới"
display(prov)

In [ ]:
order = counts.sort_values(["train", "tổng"]).index
fig, ax = plt.subplots(figsize=(8.5, 0.24 * len(CLASSES) + 1.6))
left = np.zeros(len(order))
for s, col in zip(SPLITS, [C["blue"], C["orange"], C["aqua"]]):
    vals = counts.loc[order, s].values
    ax.barh(order, vals, left=left, height=0.62, color=col, edgecolor=C["surface"], linewidth=1.5, label=s)
    left += vals
for y, v in enumerate(left):
    ax.text(v + left.max() * 0.01, y, f"{int(v):,}", va="center", fontsize=8, color=C["text2"])
ax.set_xlim(0, left.max() * 1.08); ax.set_ylim(-0.7, len(order) - 0.3)
ax.grid(axis="y", visible=False)
ax.set_xlabel("số ảnh")
ax.set_title("Số ảnh mỗi lớp sau khi chia lại (xếp theo số ảnh train)", pad=30)
ax.legend(ncol=3, loc="lower left", bbox_to_anchor=(0, 1.0), borderaxespad=0.2, handlelength=1.2)
save(fig, "pre02_class_distribution_after_split")

> **Nhận xét**
>
> - Sau khi chia lại: **train 6.109 / valid 1.306 / test 1.306** (70,0 / 15,0 / 15,0%).
> - Mỗi bang có **14–28 ảnh test** (trung bình 23,3) thay vì 5 ảnh như bộ gốc. Với 1.306 ảnh test, sai số của accuracy tổng giảm từ khoảng ±5,6 xuống khoảng ±2,6 điểm phần trăm (khoảng tin cậy 95%, quanh mức accuracy 65%).
> - Đổi lại, train còn 68–129 ảnh mỗi bang (trung bình 109,1) thay vì 86–175 (trung bình 146). Mất khoảng một phần tư dữ liệu train là cái giá để có kết quả đánh giá đáng tin.
> - Bảng nguồn gốc cho thấy ảnh đã được trộn thật sự: tập test mới lấy 1.217 ảnh từ `train` gốc, 49 từ `valid` gốc và 40 từ `test` gốc.
> - 693 ảnh thuộc các nhóm ảnh giống nhau nằm hết ở train. Vì các ảnh này không được bốc ngẫu nhiên, valid và test hơi thiếu những mẫu biển rất phổ biến; rõ nhất là Wyoming (56 trong 113 ảnh train của bang này). Khi đọc recall của bang này cần nhớ điều đó.
> - Mức lệch giữa các lớp vẫn nhẹ (lớp ít nhất là CNMI với 68 ảnh train), nên không cần oversampling; dùng F1 macro bên cạnh accuracy.

## 5. Ghi bộ dữ liệu sạch và kiểm tra

Ảnh được **sao chép** sang `CLEAN_DIR/<tập>/<bang>/`, với tên bang đã sửa. Tên file mới có tiền tố là thư mục gốc (ví dụ `train_001.jpg`, `valid_3.jpg`) để không trùng tên và vẫn truy được nguồn.

File `plates.csv` mới giữ bốn cột của bản gốc (`class id`, `filepaths`, `labels`, `data set`) và thêm cột `original filepath`. Cả `filepaths` lẫn `labels` đều dùng tên bang đã sửa.

In [ ]:
if os.path.exists(CLEAN_DIR):
    shutil.rmtree(CLEAN_DIR)                      # tạo lại từ đầu để không lẫn file của lần chạy trước
man = []
for r in tqdm(kept.itertuples(), total=len(kept), desc="sao chép"):
    new_name = f"{r.split}_{r.name}"
    d = os.path.join(CLEAN_DIR, r.split_new, r.label_new)
    os.makedirs(d, exist_ok=True)
    shutil.copyfile(r.path, os.path.join(d, new_name))
    man.append((CLASSES.index(r.label_new), f"{r.split_new}/{r.label_new}/{new_name}", r.label_new, r.split_new, r.key))
man = pd.DataFrame(man, columns=["class id", "filepaths", "labels", "data set", "original filepath"])
man["_o"] = man["data set"].map({"train": 0, "valid": 1, "test": 2})
man = man.sort_values(["_o", "class id", "filepaths"]).drop(columns="_o").reset_index(drop=True)
man.to_csv(os.path.join(CLEAN_DIR, "plates.csv"), index=False)
print("Đã ghi:", CLEAN_DIR)
display(man.head(3))

In [ ]:
# ---- kiểm tra bộ vừa ghi
def full(p):                                               # đường dẫn trong CSV dùng dấu "/"
    return os.path.join(CLEAN_DIR, *p.split("/"))

on_disk = {s: sum(len(fs) for _, _, fs in os.walk(os.path.join(CLEAN_DIR, s))) for s in SPLITS}
assert on_disk == total.to_dict(), (on_disk, total.to_dict())
for s in SPLITS:                                           # cả 56 bang có mặt ở mọi tập
    assert sorted(os.listdir(os.path.join(CLEAN_DIR, s))) == CLASSES, s
assert all(os.path.exists(full(p)) for p in man["filepaths"])               # CSV khớp với đĩa
assert (man["labels"] == man["filepaths"].str.split("/").str[1]).all()      # nhãn khớp tên thư mục
bad_names = [n for n in LABEL_FIXES if (man["labels"] == n).any() or man["filepaths"].str.contains(f"/{n}/").any()]
assert not bad_names

md5 = {}                                                   # các file giống hệt nhau
for p in man["filepaths"]:
    with open(full(p), "rb") as f:
        md5.setdefault(hashlib.md5(f.read()).hexdigest(), []).append(p)
same_files = [v for v in md5.values() if len(v) > 1]    # file giống hệt nhau (nếu có) phải cùng một tập
n_identical = len(same_files)
assert all(len({p.split("/")[0] for p in v}) == 1 for v in same_files)

new_split = dict(zip(kept["key"], kept["split_new"]))      # không cặp giống nhau nào bị tách ra hai tập
straddle = sum(new_split[r.image_a] != new_split[r.image_b] for r in pairs.itertuples())
assert straddle == 0

raw_now = sum(f.lower().endswith((".jpg", ".jpeg", ".png"))        # bộ gốc không bị đụng tới
              for s in SPLITS for _, _, fs in os.walk(os.path.join(RAW_DIR, s)) for f in fs)
assert raw_now == len(df)
check = pd.Series({
    "số ảnh trong bộ mới": f"{sum(on_disk.values()):,}  {on_disk}",
    "số lớp ở mỗi tập": f"{len(CLASSES)} / {len(CLASSES)} / {len(CLASSES)}",
    "dòng CSV trỏ tới file không tồn tại": 0,
    "dòng CSV có nhãn khác tên thư mục": 0,
    "tên lớp sai chính tả còn lại (thư mục và CSV)": len(bad_names),
    "nhóm file giống hệt nhau (đều nằm chung một tập)": n_identical,
    "cặp ảnh giống nhau nằm ở hai tập khác nhau": f"{straddle} / {len(pairs)}",
    "số ảnh trong thư mục bộ gốc (không đổi)": f"{raw_now:,}",
}, name="kết quả")
display(check.to_frame())

> **Nhận xét**
>
> - Bộ mới có 8.721 ảnh, đủ 56 bang ở cả ba tập. `plates.csv` mới khớp hoàn toàn với ảnh trên đĩa.
> - Không còn tên `HAWAI` hay `MISSIPPI` ở bất kỳ đâu: cả tên thư mục, cột `labels` và cột `filepaths` đều dùng tên đúng.
> - Không cặp ảnh giống nhau nào bị tách ra hai tập khác nhau (0 / 485 cặp). Cặp giống hệt từng pixel ở Oregon nằm chung trong train.
> - Bộ gốc vẫn nguyên 8.721 ảnh.

## 6. Normalize

Tính mean và std từng kênh màu **chỉ trên train mới** (không dùng valid và test, để không rò rỉ thông tin). Hai mô hình CNN tự thiết kế dùng các giá trị này. Mô hình transfer learning thì **không** dùng chúng: mỗi mạng pretrained của Keras có hàm `preprocess_input` riêng, phải dùng đúng hàm đó để khớp với trọng số đã huấn luyện sẵn.

In [ ]:
s1, s2, n = np.zeros(3), np.zeros(3), 0
for p in tqdm(man.loc[man["data set"] == "train", "filepaths"], desc="mean/std"):
    a = np.asarray(PILImage.open(full(p)).convert("RGB"), dtype=np.float64).reshape(-1, 3) / 255.0
    s1 += a.sum(0); s2 += (a ** 2).sum(0); n += len(a)
MEAN = [round(float(v), 4) for v in s1 / n]
STD = [round(float(v), 4) for v in np.sqrt(s2 / n - (s1 / n) ** 2)]
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
display(pd.DataFrame({"mean (train mới)": MEAN, "std (train mới)": STD,
                      "mean ImageNet": IMAGENET_MEAN, "std ImageNet": IMAGENET_STD}, index=["R", "G", "B"]))

> **Nhận xét**
>
> - mean = **[0,548; 0,540; 0,507]**, std = **[0,292; 0,274; 0,284]**. Các giá trị này gần như trùng với giá trị EDA đo trên toàn bộ ảnh (mean [0,550; 0,541; 0,506]), tức là việc chia lại không làm lệch phân phối pixel.
> - So với ImageNet, ảnh biển số sáng hơn và có độ phân tán lớn hơn (nhiều vùng rất sáng và rất tối do chữ trên nền), nên dùng mean / std riêng cho các mô hình train từ đầu là hợp lý.
> - Ảnh trên đĩa không bị đổi. Normalize được áp dụng lúc nạp dữ liệu (bước 8): `giá trị mới = (pixel / 255 − mean) / std`, tính riêng từng kênh.

## 7. Augment

Các phép biến đổi được chọn theo kết quả EDA, dùng các lớp tiền xử lý có sẵn của Keras:

| Phép biến đổi | Lớp Keras | Mức | Lý do |
|---|---|---|---|
| Xoay | `RandomRotation` | ±8° | Ảnh trung bình của mọi lớp cho thấy biển luôn thẳng; ảnh thật chỉ nghiêng nhẹ |
| Dịch | `RandomTranslation` | ≤ 6% mỗi chiều | Biển luôn canh giữa; tên bang nằm sát mép nên không dịch nhiều |
| Co giãn | `RandomZoom` | ±10% | Mô phỏng khung cắt rộng hoặc hẹp hơn một chút |
| Độ sáng | `RandomBrightness` | ±10% thang 0–255 (khoảng ±25 mức) | Trong cùng một bang độ sáng đã thay đổi rất nhiều, nên không phải tín hiệu phân loại |
| Tương phản | `RandomContrast` | ±20% | Như trên |

**Không dùng:** lật ngang / lật dọc (chữ bị ngược), random crop mạnh (cắt mất tên bang), đổi hue hoặc chuyển xám (màu là đặc trưng của một số bang như Vermont, Delaware, New Mexico).

Augment chỉ áp dụng cho **train**; valid và test giữ nguyên ảnh.

In [ ]:
FILL = float(round(255 * np.mean(MEAN)))          # tô phần viền lộ ra bằng mức xám trung bình của bộ dữ liệu

augment = keras.Sequential([
    layers.RandomRotation(8 / 360, fill_mode="constant", fill_value=FILL),
    layers.RandomTranslation(0.06, 0.06, fill_mode="constant", fill_value=FILL),
    layers.RandomZoom((-0.1, 0.1), fill_mode="constant", fill_value=FILL),
    layers.RandomBrightness(0.1, value_range=(0, 255)),
    layers.RandomContrast(0.2),
], name="augment")

def apply_augment(x):
    # x: ảnh float32 thang 0-255, dạng (batch, cao, rộng, 3)
    return tf.clip_by_value(augment(x, training=True), 0.0, 255.0)

sample = man[man["data set"] == "train"].sample(4, random_state=SEED)
imgs, caps = [], []
for r in sample.itertuples():
    im = PILImage.open(full(r.filepaths)).convert("RGB").resize((IMG_W, IMG_H))
    imgs.append(im); caps.append(f"ảnh gốc\n{r.labels}")
    x = tf.convert_to_tensor(np.asarray(im, dtype=np.float32)[None])
    for k in range(5):
        out = apply_augment(x)[0].numpy().astype(np.uint8)
        imgs.append(PILImage.fromarray(out)); caps.append(f"augment {k + 1}\n")
show_grid(imgs, caps, 6, "Ảnh gốc (cột đầu) và 5 phiên bản augment", "pre03_augmentation_preview", cell=2.0)

> **Nhận xét**
>
> - Sau augment, **chữ và tên bang vẫn đọc được** ở các phiên bản: các phép biến đổi đủ nhẹ để không phá tín hiệu chính.
> - Phần viền lộ ra khi xoay hoặc dịch được tô bằng mức xám trung bình của bộ dữ liệu thay vì màu đen, để sau khi normalize vùng đó có giá trị gần 0 và mô hình ít bị chú ý vào viền.
> - Các lớp chuẩn của Keras không có phép biến dạng phối cảnh và làm mờ ở mọi phiên bản, nên pipeline này không dùng hai phép đó (EDA xếp chúng vào loại "có thể thêm").
> - Nếu mô hình vẫn overfit mạnh (accuracy train cao hơn hẳn valid), có thể tăng nhẹ mức xoay và dịch; không nên thêm phép lật.

## 8. Nạp dữ liệu bằng `tf.data`

`image_dataset_from_directory` đọc ảnh và lấy nhãn từ tên thư mục của bộ sạch. Thứ tự lớp là thứ tự bảng chữ cái và giống nhau ở cả ba tập (truyền `class_names=CLASSES` để cố định).

Hàm `make_datasets` trả về ba `tf.data.Dataset`:

- **train:** xáo trộn → augment → normalize.
- **valid, test:** chỉ normalize.

Tham số `normalize_inputs`:

- `True` (mặc định): chuẩn hoá bằng mean / std của bộ dữ liệu. Dùng cho **hai CNN tự thiết kế**.
- `False`: giữ ảnh ở thang 0–255. Dùng cho **transfer learning**, khi đó áp dụng hàm `preprocess_input` của chính mạng pretrained (ví dụ `keras.applications.resnet50.preprocess_input`).

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
MEAN_255 = tf.constant([m * 255 for m in MEAN], dtype=tf.float32)
STD_255 = tf.constant([v * 255 for v in STD], dtype=tf.float32)

def normalize(x):
    # (pixel - mean*255) / (std*255) = (pixel/255 - mean) / std, tính riêng từng kênh
    return (x - MEAN_255) / STD_255

def load_split(split, shuffle, batch_size=BATCH_SIZE):
    return keras.utils.image_dataset_from_directory(
        os.path.join(CLEAN_DIR, split), labels="inferred", label_mode="int", class_names=CLASSES,
        image_size=(IMG_H, IMG_W), batch_size=batch_size, shuffle=shuffle, seed=SEED)

def make_datasets(normalize_inputs=True, batch_size=BATCH_SIZE):
    def prep(training):
        def f(x, y):
            x = tf.cast(x, tf.float32)
            if training:
                x = apply_augment(x)
            if normalize_inputs:
                x = normalize(x)
            return x, y
        return f
    return {
        "train": load_split("train", True, batch_size).map(prep(True), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE),
        "valid": load_split("valid", False, batch_size).map(prep(False), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE),
        "test": load_split("test", False, batch_size).map(prep(False), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE),
    }

dsets = make_datasets()
n_batches = {k: int(v.cardinality().numpy()) for k, v in dsets.items()}
x, y = next(iter(dsets["train"]))
print("Số batch:", n_batches, "· số ảnh:", total.to_dict())
print("Một batch train:", tuple(x.shape), x.dtype.name, "· nhãn:", tuple(y.shape), y.dtype.name)
print(f"Giá trị sau normalize (một batch train): min {float(tf.reduce_min(x)):.2f} · max {float(tf.reduce_max(x)):.2f}"
      f" · mean {float(tf.reduce_mean(x)):.2f} · std {float(tf.math.reduce_std(x)):.2f}")

# kiểm tra: toàn bộ valid sau normalize phải có trung bình gần 0 và độ lệch chuẩn gần 1 ở mỗi kênh
s1 = tf.zeros(3); s2 = tf.zeros(3); n = 0
for xb, _ in dsets["valid"]:
    s1 += tf.reduce_sum(xb, axis=(0, 1, 2)); s2 += tf.reduce_sum(xb ** 2, axis=(0, 1, 2)); n += int(tf.size(xb)) // 3
m = (s1 / n).numpy(); sd = np.sqrt((s2 / n).numpy() - m ** 2)
print("Valid sau normalize — mean theo kênh:", [round(float(v), 2) for v in m], "· std theo kênh:", [round(float(v), 2) for v in sd])

config = {"framework": "tensorflow/keras", "data_dir": CLEAN_DIR, "classes": CLASSES, "num_classes": len(CLASSES),
          "img_size_hw": [IMG_H, IMG_W], "input_shape": [IMG_H, IMG_W, 3], "mean": MEAN, "std": STD,
          "splits": {k: int(v) for k, v in total.items()}, "seed": SEED, "batch_size": BATCH_SIZE,
          "removed_images": 0, "similar_threshold": GROUP_T, "images_forced_to_train": int(n_forced), "label_fixes": LABEL_FIXES}
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
print("Đã ghi", CONFIG_PATH)

## Tổng kết

| Hạng mục | Trước tiền xử lý (bộ gốc) | Sau tiền xử lý |
|---|---|---|
| Số ảnh | 8.721 | 8.721 (không xoá ảnh nào) |
| train / valid / test | 8.161 / 280 / 280 | 6.109 / 1.306 / 1.306 |
| Ảnh test mỗi bang | 5 | 14–28 |
| Cặp ảnh giống nhau nằm ở hai tập khác nhau | 58 trong 485 cặp | 0 (cả 693 ảnh liên quan nằm ở train) |
| Tên lớp sai chính tả | `HAWAI`, `MISSIPPI` (thư mục và CSV) | đã sửa ở thư mục và CSV |
| Kích thước đầu vào | 224×128 | `(128, 224, 3)` — cao × rộng × kênh, giữ nguyên tỉ lệ |
| Normalize | – | mean [0,548; 0,540; 0,507], std [0,292; 0,274; 0,284] |
| Augment (train) | – | xoay ±8°, dịch ≤ 6%, co giãn ±10%, độ sáng ±25 mức, tương phản ±20% |

**Dùng lại ở các notebook mô hình**

- Hai CNN tự thiết kế: `dsets = make_datasets()`, đầu vào của mô hình là `keras.Input(shape=(128, 224, 3))`, rồi `model.fit(dsets["train"], validation_data=dsets["valid"])`.
- Transfer learning: `dsets = make_datasets(normalize_inputs=False)` và đặt `preprocess_input` của mạng pretrained làm lớp đầu tiên của mô hình.
- Nhãn ở dạng số nguyên, nên dùng loss `sparse_categorical_crossentropy`.
- `data/pre_outputs/preprocess_config.json` lưu mean, std, danh sách lớp và kích thước ảnh.

**Giới hạn**

- Trong train còn một số ảnh lặp lại, vì ảnh trùng được giữ chứ không xoá.
- Cách tìm ảnh giống nhau bắt được ảnh bị lưu lại, cắt lại hoặc đổi màu; hai bức ảnh khác nhau chụp cùng một tấm biển có thể không bị phát hiện và vẫn có thể nằm ở hai tập khác nhau.
- Ngưỡng 0,70 được chọn thực nghiệm trong EDA, bằng cách xem các cặp quanh ngưỡng. Nó chỉ quyết định ảnh nào phải đi chung vào train chứ không xoá ảnh nào, nên đặt hơi thấp cũng không làm mất dữ liệu.